In [5]:
import ollama
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_ollama import OllamaEmbeddings

from chromadb.config import Settings
from chromadb import Client
from langchain_chroma import Chroma
import gradio as gr
import re
from concurrent.futures import ThreadPoolExecutor

In [2]:
#. Step 1: Load the document using PyMuPDFLoader
loader = PyMuPDFLoader("Xiao2025.pdf")
documents = loader.load()

#. Step 2: Split text into smaller chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

#. Step 3: Initialize Ollama embeddings
embedding_function = OllamaEmbeddings(model="deepseek-r1:1.5b")


#. Step 4: Parallelize embedding generation
def generate_embedding(chunk):
    return embedding_function.embed_query(chunk.page_content)

#. This next line takes about 7 minutes on my M1 Macbook Pro with 32GB RAM;
#.   you can tell when it's done by watching the window running `ollama serve`
with ThreadPoolExecutor() as executor:
    embeddings = list(executor.map(generate_embedding, chunks))

#. Step 5: Recreate the collection
client = Client(Settings())

In [3]:
collection = client.create_collection(name="foundations_of_llms")

#. Step 6: Add documents and embeddings to Chroma
for idx, chunk in enumerate(chunks):
    collection.add(
        documents=[chunk.page_content],
        metadatas=[{"id": idx}],
        embeddings=[embeddings[idx]],
        ids=[str(idx)],  # Ensure IDs are strings
    )

print("Embeddings stored successfully!")

Embeddings stored successfully!


In [6]:
retriever = Chroma(
    collection_name="foundations_of_llms",
    client=client,
    embedding_function=embedding_function,
).as_retriever()


def retrieve_context(question):
    results = retriever.invoke(question)
    context = "\n\n".join([doc.page_content for doc in results])
    return context

In [7]:
def query_deepseek(question, context):

    # Format the input as a structured prompt
    formatted_promt = f"Question: {question}\n\nContext: {context}"

    # Send the prompt to DeepSeek-R1 using Ollama
    response = ollama.chat(
        model="deepseek-r1:1.5b", messages=[{"role": "user", "content": formatted_promt}]
    )

    # Extract and clean the response
    response_content = response["message"]["content"]
    final_answer = re.sub(
        r"<think>.*?</think>", "", response_content, flags=re.DOTALL
    ).strip()
    return final_answer

In [8]:
def rag_pipeline(question):

    # Retrieve context from the vector store
    context = retrieve_context(question)

    # Generate an answer using DeepSeek-R1
    answer = query_deepseek(question, context)
    return answer

In [9]:
def ask_question(question):
    # Run the RAG pipeline
    return rag_pipeline(question)

In [10]:
#. Create a Gradio interface
interface = gr.Interface(
    fn=ask_question,
    inputs="text",
    outputs="text",
    title="RAG Chatbot: Foundations of LLMs",
    description="Ask any question about the Foundations of LLMs book. Powered by DeepSeek-R1.",
)

In [ ]:
interface.launch(debug=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Created dataset file at: .gradio/flagged/dataset1.csv
